# 去筆跡模型：Google Colab GPU 訓練\n\n**怎麼用：打開 → 執行階段 → 變更執行階段類型選 GPU → 全部執行。**\n\n本筆記本會從 Drive 讀取最新訓練資料，使用 GPU 訓練，並把模型存回 Drive；不會修改 Git repo。

## 1. 掛載 Google Drive\n授權後，資料位置是 `MyDrive/OCR訓練資料`。

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\nDRIVE = '/content/drive/MyDrive/OCR訓練資料'


## 2. 確認 GPU\n若沒有顯示 NVIDIA GPU，請回到「變更執行階段類型」選 GPU 後，重新全部執行。

In [ ]:
!nvidia-smi


## 3. 解開最新資料與準備程式\n選取 Drive 中檔名最新的 `dehand_data_*.tar`，解到 `/content`；解壓後會保留 `ocr/dehand/data` 路徑。

In [ ]:
import glob, os, shutil, tarfile\narchives = sorted(glob.glob(f'{DRIVE}/dehand_data_*.tar'))\nassert archives, 'Drive 找不到 dehand_data_*.tar，請先在 Mac 執行 pack_data.py'\narchive = archives[-1]\nprint('使用資料：', archive)\nwith tarfile.open(archive) as tf:\n    tf.extractall('/content', filter='data')\nos.makedirs('/content/ocr/dehand', exist_ok=True)\nfor name in ('train.py', 'pairs_build.py'):\n    shutil.copy2(f'{DRIVE}/code/{name}', f'/content/ocr/dehand/{name}')\nprint('資料與程式已就緒。')


## 4. 安裝訓練所需套件\n安裝 PyTorch、ONNX 匯出與 OpenCV。Colab 已有 CUDA 驅動；下一格會確認 PyTorch 看得到 CUDA。

In [ ]:
!pip -q install torch onnx onnxscript opencv-python-headless\nimport torch\nassert torch.cuda.is_available(), 'PyTorch 沒有偵測到 CUDA，請確認執行階段已選 GPU'\nprint('PyTorch CUDA：', torch.cuda.get_device_name(0))


## 5. 在 Colab 工作副本啟用 CUDA\n原始 `train.py` 僅選 MPS/CPU；此處只補丁 `/content` 內的副本，優先選 CUDA，完全不修改 Drive 或 repo 的原檔。

In [ ]:
from pathlib import Path\ntrain = Path('/content/ocr/dehand/train.py')\nold = 'dev = torch.device("mps" if torch.backends.mps.is_available() else "cpu")'\nnew = 'dev = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))'\ntext = train.read_text(encoding='utf-8')\nassert old in text, 'train.py 裝置選擇格式已變，請先更新這個筆記本的補丁'\ntrain.write_text(text.replace(old, new, 1), encoding='utf-8')\nprint('已讓 Colab 工作副本優先使用 CUDA。')


## 6. 訓練\n使用 60 個 epoch、每 epoch 300 批、batch 32；結果會輸出至 `/content/run`。請勿中斷此格。

In [ ]:
!cd /content && python ocr/dehand/train.py --data ocr/dehand/data --epochs 60 --steps 300 --batch 32 --out /content/run


## 7. 把模型存回 Drive\n以目前時間建立唯一版本資料夾，複製 `dehand.onnx`、可能存在的外部權重 `dehand.onnx.data`，以及評估結果 `dehand.json`。

In [ ]:
from datetime import datetime\nmodel_dir = f'{DRIVE}/models/{datetime.now():%Y%m%d-%H%M%S}'\nos.makedirs(model_dir, exist_ok=False)\nfor name in ('dehand.onnx', 'dehand.onnx.data', 'dehand.json'):\n    source = f'/content/run/{name}'\n    if os.path.exists(source):\n        shutil.copy2(source, f'{model_dir}/{name}')\nassert os.path.exists(f'{model_dir}/dehand.onnx') and os.path.exists(f'{model_dir}/dehand.json')\nprint('已存回：', model_dir)\nprint(Path(f'{model_dir}/dehand.json').read_text(encoding='utf-8'))
